In [2]:
# Librerías
import pandas as pd
from pathlib import Path
import ee

/home/ddayann/miniconda3/envs/tf_cuda_gee/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


#### Mecanismo de autorización
1. Al ejecutar las siguientes líneas se habilitará un link de acceso
2. Asegura los permisos apropiados para google Earth Engine y demás necesarios
3. Se genera un Token
4. Copiar y pegar el token en la ventana emergente que aparece en la ventana superior de VSC

* Para habilitar nuevos usuarios es necesario desmarcar "ee.Authenticate(force=True)" - Actualmente ya tiene habilitado el ingreso del usuario presente.  
(4/1AeoWuM9_9T1gCVAvTELP8S2YXcFW26H8_p7npV3xK4wc_OUw6Fb5i68ThS8)

In [ ]:
# Log Google Earth Engine
# ee.Authenticate(force=True)
ee.Initialize(project='ee-cafe-494102')


Successfully saved authorization token.


In [ ]:
# Cargar asset de municipios
municipios_ee = ee.FeatureCollection("projects/ee-cafe-494102/assets/municipios")

# Validación
print("Número de municipios:", municipios_ee.size().getInfo())
print("Columnas:", municipios_ee.first().propertyNames().getInfo())

Número de municipios: 27
Columnas: ['OBJECTID', 'MpNombre', 'Depto', 'MpCodigo', 'MpCategor', 'SHAPE_Leng', 'MpAltitud', 'SHAPE_Area', 'MpArea', 'system:index', 'MpNorma', 'Restriccio']


In [4]:
# Fechas de análisis
fecha_inicio = "2005-01-01"
fecha_fin = "2025-12-31"

In [5]:
# Colección MODIS NDVI
ndvi_modis = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate(fecha_inicio, fecha_fin)
    .select("NDVI")
)

# Escalar NDVI
def escalar_ndvi(image):
    ndvi = image.multiply(0.0001).rename("ndvi")
    return ndvi.copyProperties(image, ["system:time_start", "system:index"])

ndvi_modis = ndvi_modis.map(escalar_ndvi)

print("Número de imágenes MODIS:", ndvi_modis.size().getInfo())

Número de imágenes MODIS: 483


In [8]:
# NDVI por municipio - versión optimizada para CSV liviano

campo_municipio = "MpNombre"

def ndvi_por_municipio(image):
    fecha = ee.Date(image.get("system:time_start")).format("YYYY-MM-dd")
    
    stats = image.reduceRegions(
        collection=municipios_ee,
        reducer=(
            ee.Reducer.mean()
            .combine(ee.Reducer.median(), sharedInputs=True)
            .combine(ee.Reducer.min(), sharedInputs=True)
            .combine(ee.Reducer.max(), sharedInputs=True)
            .combine(ee.Reducer.stdDev(), sharedInputs=True)
            .combine(ee.Reducer.count(), sharedInputs=True)
        ),
        scale=250,
        tileScale=4
    )
    
    return stats.map(
        lambda f: ee.Feature(
            None,
            {
                "municipio": f.get(campo_municipio),
                "fecha": fecha,
                "ndvi_mean": f.get("mean"),
                "ndvi_median": f.get("median"),
                "ndvi_min": f.get("min"),
                "ndvi_max": f.get("max"),
                "ndvi_stdDev": f.get("stdDev"),
                "n_pixeles": f.get("count"),
                "producto": "MODIS/061/MOD13Q1"
            }
        )
    )

ndvi_municipal_limpio = ndvi_modis.map(
    ndvi_por_municipio
).flatten()

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=ndvi_municipal_limpio,
    description="ndvi_municipal_mod13q1_2005_2025",
    folder="GEE_exports",
    fileNamePrefix="ndvi_municipal_mod13q1_2005_2025",
    fileFormat="CSV",
    selectors=[
        "municipio",
        "fecha",
        "ndvi_mean",
        "ndvi_median",
        "ndvi_min",
        "ndvi_max",
        "ndvi_stdDev",
        "n_pixeles",
        "producto"
    ]
)

task.start()

print("Exportación iniciada.")
print("Task ID:", task.id)

Exportación iniciada.
Task ID: OISWWRAYAAL6XZLI5OVI6CZK


#### Validación del proceso 
A fin de validar que el proceso se está ejecutando es necesario abrir el proyecto de Google Earth Engine, en la pestaña Task. Es posible ver el avance del proceso.  
Al final del procesamiento el archivo será guardado en Google Drive: "/MyDrive/Gee_exports"

In [3]:
# Lectura archivo CSV exportado
BASE_DIR = Path.cwd().parents[1]
ruta_ndvi = BASE_DIR / "data" / "raw" / "ndvi_municipal_mod13q1_2005_2025.csv"
df_ndvi = pd.read_csv(ruta_ndvi)

df_ndvi

,municipio,fecha,ndvi_mean,ndvi_median,ndvi_min,ndvi_max,ndvi_stdDev,n_pixeles,producto
0,Villamaria,2005-01-01,0.468332,0.504226,-0.0763,0.9986,0.259046,7586,MODIS/061/MOD13Q1
1,Risaralda,2005-01-01,0.669925,0.744011,0.0771,0.9968,0.189740,1609,MODIS/061/MOD13Q1
2,Neira,2005-01-01,0.603923,0.701356,0.0420,0.9959,0.228572,6262,MODIS/061/MOD13Q1
3,Anserma,2005-01-01,0.618185,0.697465,0.0479,0.9950,0.207892,3589,MODIS/061/MOD13Q1
4,Manzanares,2005-01-01,0.674509,0.787148,0.0517,0.9981,0.235472,3361,MODIS/061/MOD13Q1
...,...,...,...,...,...,...,...,...,...
13036,Belalcázar,2025-12-19,0.698339,0.767522,0.0544,0.9874,0.176198,1960,MODIS/061/MOD13Q1
13037,San José,2025-12-19,0.695235,0.776057,0.0634,0.9595,0.192576,1097,MODIS/061/MOD13Q1
13038,Marquetalia,2025-12-19,0.845785,0.853701,0.1691,0.9577,0.046625,1589,MODIS/061/MOD13Q1
13039,Norcasia,2025-12-19,0.789316,0.822285,0.0070,0.9929,0.131120,3900,MODIS/061/MOD13Q1


In [4]:
df_ndvi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13041 entries, 0 to 13040
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   municipio    13041 non-null  object 
 1   fecha        13041 non-null  object 
 2   ndvi_mean    13041 non-null  float64
 3   ndvi_median  13041 non-null  float64
 4   ndvi_min     13041 non-null  float64
 5   ndvi_max     13041 non-null  float64
 6   ndvi_stdDev  13041 non-null  float64
 7   n_pixeles    13041 non-null  int64  
 8   producto     13041 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 917.1+ KB


In [5]:
# Tipificación
df_ndvi["municipio"] = df_ndvi["municipio"].astype("category")
df_ndvi["fecha"] = pd.to_datetime(df_ndvi["fecha"])
df_ndvi = df_ndvi.drop(columns=["producto"])

In [6]:
df_ndvi

,municipio,fecha,ndvi_mean,ndvi_median,ndvi_min,ndvi_max,ndvi_stdDev,n_pixeles
0,Villamaria,2005-01-01,0.468332,0.504226,-0.0763,0.9986,0.259046,7586
1,Risaralda,2005-01-01,0.669925,0.744011,0.0771,0.9968,0.189740,1609
2,Neira,2005-01-01,0.603923,0.701356,0.0420,0.9959,0.228572,6262
3,Anserma,2005-01-01,0.618185,0.697465,0.0479,0.9950,0.207892,3589
4,Manzanares,2005-01-01,0.674509,0.787148,0.0517,0.9981,0.235472,3361
...,...,...,...,...,...,...,...,...
13036,Belalcázar,2025-12-19,0.698339,0.767522,0.0544,0.9874,0.176198,1960
13037,San José,2025-12-19,0.695235,0.776057,0.0634,0.9595,0.192576,1097
13038,Marquetalia,2025-12-19,0.845785,0.853701,0.1691,0.9577,0.046625,1589
13039,Norcasia,2025-12-19,0.789316,0.822285,0.0070,0.9929,0.131120,3900


In [7]:
df_ndvi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13041 entries, 0 to 13040
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   municipio    13041 non-null  category      
 1   fecha        13041 non-null  datetime64[ns]
 2   ndvi_mean    13041 non-null  float64       
 3   ndvi_median  13041 non-null  float64       
 4   ndvi_min     13041 non-null  float64       
 5   ndvi_max     13041 non-null  float64       
 6   ndvi_stdDev  13041 non-null  float64       
 7   n_pixeles    13041 non-null  int64         
dtypes: category(1), datetime64[ns](1), float64(5), int64(1)
memory usage: 727.3 KB


In [8]:
# Agregación mensual
df_ndvi["mes"] = df_ndvi["fecha"].dt.month
df_ndvi["anio"] = df_ndvi["fecha"].dt.year

ndvi_mes = (
    df_ndvi
    .groupby(["municipio", "anio", "mes"])
    .agg({
        "ndvi_mean": "mean",
        "ndvi_median": "mean",
        "ndvi_min": "min",
        "ndvi_max": "max",
        "ndvi_stdDev": "mean",
        "n_pixeles": "mean"
    })
    .reset_index()
)

# Crear fecha base (primer día del mes)
ndvi_mes["date"] = pd.to_datetime(
    ndvi_mes["anio"].astype(str) + "-" + ndvi_mes["mes"].astype(str) + "-01"
)

# Llevar al último día del mes
ndvi_mes["date"] = ndvi_mes["date"] + pd.offsets.MonthEnd(0)

ndvi_mes = ndvi_mes[["municipio", "date", "anio", "mes",
                    "ndvi_mean", "ndvi_median", "ndvi_min",
                    "ndvi_max", "ndvi_stdDev", "n_pixeles"]]

/tmp/ipykernel_9210/756805544.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["municipio", "anio", "mes"])


In [9]:
# Agregación anual
ndvi_anual = (
    ndvi_mes
    .groupby(["municipio", "anio"])
    .agg({
        "ndvi_mean": "mean",
        "ndvi_median": "mean",
        "ndvi_min": "min",
        "ndvi_max": "max",
        "ndvi_stdDev": "mean",
        "n_pixeles": "mean"
    })
    .reset_index()
)

# Crear date anual como cierre de año
ndvi_anual["date"] = pd.to_datetime(
    ndvi_anual["anio"].astype(str) + "-12-31"
)

# Reordenar columnas
ndvi_anual = ndvi_anual[
    ["municipio", "date", "anio",
     "ndvi_mean", "ndvi_median", "ndvi_min",
     "ndvi_max", "ndvi_stdDev", "n_pixeles"]
]

/tmp/ipykernel_9210/1536030839.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["municipio", "anio"])


In [10]:
display(ndvi_mes.head(12))
display(ndvi_anual.head(21))

,municipio,date,anio,mes,ndvi_mean,ndvi_median,ndvi_min,ndvi_max,ndvi_stdDev,n_pixeles
0,Aguadas,2005-01-31,2005,1,0.679196,0.712829,0.0297,0.9985,0.153667,7980.0
1,Aguadas,2005-02-28,2005,2,0.694763,0.732490,0.0376,0.9993,0.146363,7981.0
2,Aguadas,2005-03-31,2005,3,0.735368,0.757732,0.0487,0.9991,0.113620,7981.0
3,Aguadas,2005-04-30,2005,4,0.683401,0.724797,0.0158,0.9988,0.147255,7981.0
4,Aguadas,2005-05-31,2005,5,0.756183,0.785111,0.0261,0.9982,0.124472,7981.0
5,Aguadas,2005-06-30,2005,6,0.761704,0.783243,0.0484,0.9994,0.110209,7981.0
6,Aguadas,2005-07-31,2005,7,0.726913,0.734427,0.0533,0.9966,0.084413,7981.0
7,Aguadas,2005-08-31,2005,8,0.690223,0.724594,0.0315,0.9994,0.139671,7981.0
8,Aguadas,2005-09-30,2005,9,0.690467,0.732597,0.0174,0.9990,0.144927,7981.0
9,Aguadas,2005-10-31,2005,10,0.582588,0.631313,0.0259,0.9929,0.213653,7981.0


,municipio,date,anio,ndvi_mean,ndvi_median,ndvi_min,ndvi_max,ndvi_stdDev,n_pixeles
0,Aguadas,2005-12-31,2005,0.703595,0.738350,0.0050,0.9994,0.141924,7979.958333
1,Aguadas,2006-12-31,2006,0.691045,0.731616,0.0121,0.9987,0.150294,7979.500000
2,Aguadas,2007-12-31,2007,0.687805,0.731916,0.0089,0.9991,0.158511,7980.083333
3,Aguadas,2008-12-31,2008,0.678829,0.729487,0.0087,0.9992,0.165530,7979.791667
4,Aguadas,2009-12-31,2009,0.705580,0.748844,0.0000,0.9992,0.147902,7979.750000
5,Aguadas,2010-12-31,2010,0.655058,0.693308,0.0142,0.9991,0.170307,7979.958333
6,Aguadas,2011-12-31,2011,0.685988,0.732728,-0.0039,0.9995,0.163403,7979.750000
7,Aguadas,2012-12-31,2012,0.709586,0.744113,0.0107,0.9989,0.138576,7979.958333
8,Aguadas,2013-12-31,2013,0.713570,0.756024,0.0135,0.9991,0.142734,7979.916667
9,Aguadas,2014-12-31,2014,0.695570,0.747042,0.0114,0.9989,0.159189,7979.750000


In [11]:
# Exportar archivos finales
ruta_salida_ano = BASE_DIR / "data" / "processed" / "ndvi_caldas_anio_2005-2025.csv"
ndvi_anual.to_csv(ruta_salida_ano, index=False)
ruta_salida_mes = BASE_DIR / "data" / "processed" / "ndvi_caldas_mes_2005-2025.csv"
ndvi_mes.to_csv(ruta_salida_mes, index=False)